In [2]:
# ============================================================
# NOTEBOOK 05: EXPLORATORY DATA ANALYSIS
# Project: Credit Portfolio Risk & Expected Loss Modeling
# Dataset: Freddie Mac Single-Family Loan-Level Dataset
# ============================================================

# -----------------------------
# 1. Import libraries
# -----------------------------

import sys
import gc
import time
import warnings

from pathlib import Path

import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# -----------------------------
# 2. Project configuration
# -----------------------------

PROJECT_ROOT = Path(
    r"D:\DS Project\credit-portfolio-risk-modeling"
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config.paths import (
    ORIG_PARQUET_DIR,
    PERF_PARQUET_DIR,
    AUDIT_DIR,
    DUCKDB_TEMP_DIR
)

# Analytical data directory
ANALYTICAL_DIR = Path(
    r"E:\DS_project_backup\credit-risk-portfolio\processed\analytical"
)

# Event data directory
EVENT_DIR = Path(
    r"E:\DS_project_backup\credit-risk-portfolio\processed\events"
)

# EDA output directories
EDA_DIR = Path(
    r"E:\DS_project_backup\credit-risk-portfolio\processed\eda"
)

EDA_TABLE_DIR = EDA_DIR / "tables"
EDA_CHART_DIR = EDA_DIR / "charts"
EDA_DATA_DIR = EDA_DIR / "loan_level"

for folder in [
    EDA_DIR,
    EDA_TABLE_DIR,
    EDA_CHART_DIR,
    EDA_DATA_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

# -----------------------------
# 3. Dataset configuration
# -----------------------------

VINTAGES = [
    f"{year}Q{quarter}"
    for year in range(2006, 2013)
    for quarter in range(1, 5)
]

EXPECTED_LOANS = 11_128_442

# -----------------------------
# 4. DuckDB configuration
# -----------------------------

con = duckdb.connect()

con.execute("SET memory_limit = '3GB'")
con.execute("SET threads = 2")
con.execute("SET preserve_insertion_order = false")

temp_path = DUCKDB_TEMP_DIR.as_posix().replace("'", "''")

con.execute(
    f"SET temp_directory = '{temp_path}'"
)

# -----------------------------
# 5. Display configuration
# -----------------------------

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.4f}".format)

print("Notebook 05 configuration loaded successfully.")
print("DuckDB version:", duckdb.__version__)
print("Number of vintages:", len(VINTAGES))
print("Expected number of loans:", f"{EXPECTED_LOANS:,}")
print("Analytical directory:", ANALYTICAL_DIR)
print("EDA output directory:", EDA_DIR)

Notebook 05 configuration loaded successfully.
DuckDB version: 2.0.0
Number of vintages: 28
Expected number of loans: 11,128,442
Analytical directory: E:\DS_project_backup\credit-risk-portfolio\processed\analytical
EDA output directory: E:\DS_project_backup\credit-risk-portfolio\processed\eda


In [4]:
# ============================================================
# CELL 2: VERIFY INPUT DATA
# ============================================================

# Check analytical files
analytical_files = sorted(
    ANALYTICAL_DIR.glob("loan_month_*.parquet")
)

# Exclude the test file from Notebook 04
analytical_files = [
    f for f in analytical_files
    if not f.stem.endswith("_test")
]

# Check origination files
orig_files = sorted(
    ORIG_PARQUET_DIR.glob("orig_*.parquet")
)

# Check performance files
perf_files = sorted(
    PERF_PARQUET_DIR.glob("perf_*.parquet")
)

print("INPUT DATA VALIDATION")
print("-" * 50)

print("Expected vintages:", len(VINTAGES))
print("Analytical files:", len(analytical_files))
print("Origination files:", len(orig_files))
print("Performance files:", len(perf_files))

assert len(analytical_files) == 28
assert len(orig_files) == 28
assert len(perf_files) == 28

print("\nAll 28 vintages are available.")

INPUT DATA VALIDATION
--------------------------------------------------
Expected vintages: 28
Analytical files: 28
Origination files: 28
Performance files: 28

All 28 vintages are available.


In [5]:
# Inspect one analytical Parquet file

sample_file = (
    ANALYTICAL_DIR / "loan_month_2012Q4.parquet"
)

sample_schema = con.execute(
    """
    DESCRIBE
    SELECT *
    FROM read_parquet(?)
    """,
    [str(sample_file)]
).df()

display(sample_schema)

,column_name,column_type,null,key,default,extra
0,loan_identifier,VARCHAR,YES,None,None,None
1,period,VARCHAR,YES,None,None,None
2,current_actual_upb,VARCHAR,YES,None,None,None
3,current_loan_delinquency_status,VARCHAR,YES,None,None,None
4,loan_age,VARCHAR,YES,None,None,None
5,remaining_months_to_legal_maturity,VARCHAR,YES,None,None,None
6,underwriting_defect_and_major_servicing_defect...,VARCHAR,YES,None,None,None
7,modification_flag,VARCHAR,YES,None,None,None
8,zero_balance_code,VARCHAR,YES,None,None,None
9,zero_balance_effective_date,VARCHAR,YES,None,None,None


In [7]:
# ============================================================
# CELL 3: INSPECT ORIGINATION DATA
# ============================================================

sample_orig_file = (
    ORIG_PARQUET_DIR / "orig_2012Q4.parquet"
)

orig_schema = con.execute(
    """
    DESCRIBE
    SELECT *
    FROM read_parquet(?)
    """,
    [str(sample_orig_file)]
).df()

display(orig_schema)

,column_name,column_type,null,key,default,extra
0,classic_fico,VARCHAR,YES,None,None,None
1,first_payment_date,VARCHAR,YES,None,None,None
2,first_time_homebuyer_indicator,VARCHAR,YES,None,None,None
3,maturity_date,VARCHAR,YES,None,None,None
4,metropolitan_statistical_area_msa_or_metropoli...,VARCHAR,YES,None,None,None
5,mortgage_insurance_percentage_mi,VARCHAR,YES,None,None,None
6,number_of_units,VARCHAR,YES,None,None,None
7,occupancy_status,VARCHAR,YES,None,None,None
8,original_combined_loan_to_value_cltv,VARCHAR,YES,None,None,None
9,original_debt_to_income_dti_ratio,VARCHAR,YES,None,None,None


In [8]:
# ============================================================
# CELL 4: AUDIT LOAN-LEVEL OUTCOME AGGREGATION
# Sample vintage: 2012Q4
# ============================================================

sample_file = (
    ANALYTICAL_DIR / "loan_month_2012Q4.parquet"
)

# ------------------------------------------------------------
# 1. Inspect zero balance code distribution
# ------------------------------------------------------------

zero_balance_summary = con.execute(
    """
    SELECT
        zero_balance_code,
        COUNT(*) AS monthly_records,
        COUNT(DISTINCT loan_identifier) AS unique_loans,
        COUNT(actual_loss) AS records_with_actual_loss
    FROM read_parquet(?)
    WHERE NULLIF(TRIM(zero_balance_code), '') IS NOT NULL
    GROUP BY zero_balance_code
    ORDER BY monthly_records DESC
    """,
    [str(sample_file)]
).df()

display(zero_balance_summary)


# ------------------------------------------------------------
# 2. Check distinct terminal codes per loan
# ------------------------------------------------------------

terminal_code_check = con.execute(
    """
    WITH loan_codes AS (
        SELECT
            loan_identifier,
            COUNT(DISTINCT zero_balance_code)
                AS distinct_terminal_codes
        FROM read_parquet(?)
        WHERE NULLIF(TRIM(zero_balance_code), '') IS NOT NULL
        GROUP BY loan_identifier
    )
    SELECT
        COUNT(*) AS loans_with_terminal_codes,
        SUM(
            CASE
                WHEN distinct_terminal_codes > 1 THEN 1
                ELSE 0
            END
        ) AS loans_with_multiple_terminal_codes,
        MAX(distinct_terminal_codes)
            AS max_terminal_codes_per_loan
    FROM loan_codes
    """,
    [str(sample_file)]
).df()

display(terminal_code_check)


# ------------------------------------------------------------
# 3. Inspect actual loss availability by terminal code
# ------------------------------------------------------------

loss_availability = con.execute(
    """
    SELECT
        zero_balance_code,
        COUNT(DISTINCT loan_identifier) AS unique_loans,
        COUNT(DISTINCT loan_identifier)
            FILTER (WHERE actual_loss IS NOT NULL)
            AS loans_with_actual_loss,
        COUNT(DISTINCT loan_identifier)
            FILTER (WHERE actual_loss IS NULL)
            AS loans_without_actual_loss
    FROM read_parquet(?)
    WHERE zero_balance_code IN ('02', '03', '09', '15')
    GROUP BY zero_balance_code
    ORDER BY zero_balance_code
    """,
    [str(sample_file)]
).df()

display(loss_availability)

,zero_balance_code,monthly_records,unique_loans,records_with_actual_loss
0,01,449615,449615,0
1,16,2058,2058,0
2,09,1094,1094,1074
3,03,994,994,983
4,02,931,931,921
5,96,797,797,0
6,15,414,414,411


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,loans_with_terminal_codes,loans_with_multiple_terminal_codes,max_terminal_codes_per_loan
0,455903,0.0000,1


,zero_balance_code,unique_loans,loans_with_actual_loss,loans_without_actual_loss
0,02,931,921,10
1,03,994,983,11
2,09,1094,1074,20
3,15,414,411,3


In [9]:
# ============================================================
# CELL 5: BUILD LOAN-LEVEL DATASET - TEST 2012Q4
# ============================================================

sample_file = (
    ANALYTICAL_DIR / "loan_month_2012Q4.parquet"
)

start_time = time.time()

# ------------------------------------------------------------
# 1. Aggregate monthly observations to loan level
# ------------------------------------------------------------

con.execute(
    """
    CREATE OR REPLACE TEMP TABLE loan_level_2012Q4 AS

    WITH loan_aggregation AS (

        SELECT
            loan_identifier,

            -- Monthly observation coverage
            COUNT(*) AS performance_months,
            MIN(period) AS first_observed_period,
            MAX(period) AS last_observed_period,

            -- Delinquency indicators
            MAX(
                CASE
                    WHEN TRY_CAST(
                        TRIM(current_loan_delinquency_status)
                        AS INTEGER
                    ) >= 1 THEN 1
                    ELSE 0
                END
            ) AS ever_30dpd,

            MAX(
                CASE
                    WHEN TRY_CAST(
                        TRIM(current_loan_delinquency_status)
                        AS INTEGER
                    ) >= 2 THEN 1
                    ELSE 0
                END
            ) AS ever_60dpd,

            MAX(
                CASE
                    WHEN TRY_CAST(
                        TRIM(current_loan_delinquency_status)
                        AS INTEGER
                    ) >= 3 THEN 1
                    ELSE 0
                END
            ) AS ever_90dpd,

            MAX(
                CASE
                    WHEN TRIM(current_loan_delinquency_status) = 'RA'
                    THEN 1
                    ELSE 0
                END
            ) AS ever_reo,

            MIN(period) FILTER (
                WHERE TRY_CAST(
                    TRIM(current_loan_delinquency_status)
                    AS INTEGER
                ) >= 3
            ) AS first_90dpd_period,

            MAX(
                TRY_CAST(
                    TRIM(current_loan_delinquency_status)
                    AS INTEGER
                )
            ) AS max_numeric_delinquency_status,

            -- Terminal event information
            MAX(zero_balance_code) FILTER (
                WHERE NULLIF(TRIM(zero_balance_code), '') IS NOT NULL
            ) AS terminal_zero_balance_code,

            MAX(period) FILTER (
                WHERE NULLIF(TRIM(zero_balance_code), '') IS NOT NULL
            ) AS terminal_reporting_period,

            MAX(zero_balance_effective_date) FILTER (
                WHERE NULLIF(TRIM(zero_balance_code), '') IS NOT NULL
            ) AS zero_balance_effective_date,

            -- Actual loss
            MAX(
                TRY_CAST(NULLIF(TRIM(actual_loss), '') AS DOUBLE)
            ) FILTER (
                WHERE zero_balance_code IN ('02', '03', '09', '15')
            ) AS actual_loss,

            COUNT(*) FILTER (
                WHERE zero_balance_code IN ('02', '03', '09', '15')
                  AND NULLIF(TRIM(actual_loss), '') IS NOT NULL
            ) AS actual_loss_records

        FROM read_parquet(?)

        GROUP BY loan_identifier
    )

    SELECT
        -- Loan identifier and vintage
        o.loan_identifier,
        '2012Q4' AS vintage,

        -- Origination characteristics
        MAX(o.classic_fico) AS classic_fico,
        MAX(o.first_payment_date) AS first_payment_date,
        MAX(o.first_time_homebuyer_indicator)
            AS first_time_homebuyer_indicator,
        MAX(o.maturity_date) AS maturity_date,
        MAX(o.mortgage_insurance_percentage_mi)
            AS mortgage_insurance_percentage_mi,
        MAX(o.number_of_units) AS number_of_units,
        MAX(o.occupancy_status) AS occupancy_status,
        MAX(o.original_combined_loan_to_value_cltv)
            AS original_combined_loan_to_value_cltv,
        MAX(o.original_debt_to_income_dti_ratio)
            AS original_debt_to_income_dti_ratio,
        MAX(o.original_upb) AS original_upb,
        MAX(o.original_loan_to_value_ltv)
            AS original_loan_to_value_ltv,
        MAX(o.original_interest_rate) AS original_interest_rate,
        MAX(o.channel) AS channel,
        MAX(o.prepayment_penalty_indicator)
            AS prepayment_penalty_indicator,
        MAX(o.amortization_type) AS amortization_type,
        MAX(o.property_state) AS property_state,
        MAX(o.property_type) AS property_type,
        MAX(o.loan_purpose) AS loan_purpose,
        MAX(o.original_loan_term) AS original_loan_term,
        MAX(o.number_of_borrowers) AS number_of_borrowers,
        MAX(o.seller_name) AS seller_name,
        MAX(o.super_conforming_flag) AS super_conforming_flag,
        MAX(o.special_eligibility_program)
            AS special_eligibility_program,
        MAX(o.harp_indicator) AS harp_indicator,
        MAX(o.property_valuation_method)
            AS property_valuation_method,
        MAX(o.interest_only_i_o_indicator)
            AS interest_only_i_o_indicator,
        MAX(o.vantagescore_4_0) AS vantagescore_4_0,

        -- Performance history
        a.performance_months,
        a.first_observed_period,
        a.last_observed_period,

        -- Delinquency indicators
        a.ever_30dpd,
        a.ever_60dpd,
        a.ever_90dpd,
        a.ever_reo,
        a.first_90dpd_period,
        a.max_numeric_delinquency_status,

        -- Terminal event
        a.terminal_zero_balance_code,
        a.terminal_reporting_period,
        a.zero_balance_effective_date,

        -- Credit outcome indicators
        CASE
            WHEN a.terminal_zero_balance_code IN ('02', '03', '09')
            THEN 1 ELSE 0
        END AS credit_related_termination,

        CASE
            WHEN a.terminal_zero_balance_code = '15'
            THEN 1 ELSE 0
        END AS whole_loan_sale,

        CASE
            WHEN a.terminal_zero_balance_code = '01'
            THEN 1 ELSE 0
        END AS voluntary_termination,

        CASE
            WHEN a.terminal_zero_balance_code = '96'
            THEN 1 ELSE 0
        END AS defect_termination,

        CASE
            WHEN a.terminal_zero_balance_code = '16'
            THEN 1 ELSE 0
        END AS reperforming_securitization,

        -- Actual loss
        a.actual_loss,

        CASE
            WHEN a.actual_loss IS NOT NULL THEN 1
            ELSE 0
        END AS actual_loss_available

    FROM read_parquet(?) o

    LEFT JOIN loan_aggregation a
        ON o.loan_identifier = a.loan_identifier

    GROUP BY
        o.loan_identifier,
        a.performance_months,
        a.first_observed_period,
        a.last_observed_period,
        a.ever_30dpd,
        a.ever_60dpd,
        a.ever_90dpd,
        a.ever_reo,
        a.first_90dpd_period,
        a.max_numeric_delinquency_status,
        a.terminal_zero_balance_code,
        a.terminal_reporting_period,
        a.zero_balance_effective_date,
        a.actual_loss,
        a.actual_loss_records
    """,
    [str(sample_file), str(sample_file)]
)

print("Loan-level aggregation completed.")
print(f"Execution time: {time.time() - start_time:.2f} seconds")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loan-level aggregation completed.
Execution time: 37.70 seconds


In [10]:
# ============================================================
# CELL 6: VALIDATE LOAN-LEVEL DATASET
# ============================================================

# 1. Basic row count and uniqueness
basic_validation = con.execute(
    """
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT loan_identifier) AS unique_loans,
        COUNT(*) - COUNT(DISTINCT loan_identifier)
            AS duplicate_excess,
        COUNT(*) FILTER (
            WHERE loan_identifier IS NULL
        ) AS missing_loan_ids
    FROM loan_level_2012Q4
    """
).df()

display(basic_validation)


# 2. Outcome distribution
outcome_summary = con.execute(
    """
    SELECT
        terminal_zero_balance_code,
        COUNT(*) AS unique_loans,
        SUM(credit_related_termination)
            AS credit_related_loans,
        SUM(whole_loan_sale)
            AS whole_loan_sales,
        SUM(voluntary_termination)
            AS voluntary_terminations,
        SUM(actual_loss_available)
            AS loans_with_actual_loss
    FROM loan_level_2012Q4
    GROUP BY terminal_zero_balance_code
    ORDER BY terminal_zero_balance_code NULLS LAST
    """
).df()

display(outcome_summary)


# 3. Delinquency indicators
delinquency_summary = con.execute(
    """
    SELECT
        COUNT(*) AS total_loans,
        SUM(ever_30dpd) AS ever_30dpd_loans,
        SUM(ever_60dpd) AS ever_60dpd_loans,
        SUM(ever_90dpd) AS ever_90dpd_loans,
        SUM(ever_reo) AS ever_reo_loans
    FROM loan_level_2012Q4
    """
).df()

display(delinquency_summary)


# 4. Actual loss summary
loss_summary = con.execute(
    """
    SELECT
        COUNT(*) FILTER (
            WHERE actual_loss IS NOT NULL
        ) AS loans_with_actual_loss,

        COUNT(*) FILTER (
            WHERE actual_loss IS NULL
              AND terminal_zero_balance_code IN ('02', '03', '09', '15')
        ) AS event_loans_without_actual_loss,

        SUM(actual_loss) AS total_actual_loss,

        AVG(actual_loss) AS average_actual_loss,

        MEDIAN(actual_loss) AS median_actual_loss,

        MIN(actual_loss) AS minimum_actual_loss,

        MAX(actual_loss) AS maximum_actual_loss

    FROM loan_level_2012Q4
    """
).df()

display(loss_summary)

,total_rows,unique_loans,duplicate_excess,missing_loan_ids
0,592713,592713,0,0


,terminal_zero_balance_code,unique_loans,credit_related_loans,whole_loan_sales,voluntary_terminations,loans_with_actual_loss
0,01,449615,0.0000,0.0000,"449,615.0000",0.0000
1,02,931,931.0000,0.0000,0.0000,921.0000
2,03,994,994.0000,0.0000,0.0000,983.0000
3,09,1094,"1,094.0000",0.0000,0.0000,"1,074.0000"
4,15,414,0.0000,414.0000,0.0000,411.0000
5,16,2058,0.0000,0.0000,0.0000,0.0000
6,96,797,0.0000,0.0000,0.0000,0.0000
7,NaN,136810,0.0000,0.0000,0.0000,0.0000


,total_loans,ever_30dpd_loans,ever_60dpd_loans,ever_90dpd_loans,ever_reo_loans
0,592713,"73,095.0000","27,333.0000","19,840.0000","1,099.0000"


,loans_with_actual_loss,event_loans_without_actual_loss,total_actual_loss,average_actual_loss,median_actual_loss,minimum_actual_loss,maximum_actual_loss
0,3389,44,"157,990,358.5200","46,618.5773","39,010.0300","-71,375.3100","401,270.5900"


In [11]:
# ============================================================
# CELL 6A: RECHECK ROW COUNT AND UNIQUENESS
# ============================================================

# Check physical row count
row_count_check = con.execute(
    """
    SELECT COUNT(*) AS total_rows
    FROM loan_level_2012Q4
    """
).df()

# Check distinct loan IDs
unique_check = con.execute(
    """
    SELECT
        COUNT(DISTINCT loan_identifier) AS unique_loans,
        COUNT(*) FILTER (
            WHERE loan_identifier IS NULL
        ) AS missing_loan_ids
    FROM loan_level_2012Q4
    """
).df()

# Check whether the table contains rows
sample_check = con.execute(
    """
    SELECT *
    FROM loan_level_2012Q4
    LIMIT 5
    """
).df()

print("ROW COUNT")
display(row_count_check)

print("UNIQUENESS")
display(unique_check)

print("SAMPLE RECORDS")
display(sample_check)

print("EXPECTED ROWS: 592,713")

ROW COUNT


,total_rows
0,592713


UNIQUENESS


,unique_loans,missing_loan_ids
0,592713,0


SAMPLE RECORDS


,loan_identifier,vintage,classic_fico,first_payment_date,first_time_homebuyer_indicator,maturity_date,mortgage_insurance_percentage_mi,number_of_units,occupancy_status,original_combined_loan_to_value_cltv,original_debt_to_income_dti_ratio,original_upb,original_loan_to_value_ltv,original_interest_rate,channel,prepayment_penalty_indicator,amortization_type,property_state,property_type,loan_purpose,original_loan_term,number_of_borrowers,seller_name,super_conforming_flag,special_eligibility_program,harp_indicator,property_valuation_method,interest_only_i_o_indicator,vantagescore_4_0,performance_months,first_observed_period,last_observed_period,ever_30dpd,ever_60dpd,ever_90dpd,ever_reo,first_90dpd_period,max_numeric_delinquency_status,terminal_zero_balance_code,terminal_reporting_period,zero_balance_effective_date,credit_related_termination,whole_loan_sale,voluntary_termination,defect_termination,reperforming_securitization,actual_loss,actual_loss_available
0,F12Q40491226,2012Q4,786,201302,N,204301,0,1,P,22,999,141000,22,3.75,R,N,FRM,NY,SF,N,360,1,"JPMORGAN CHASE BANK, N.A.",N,None,Y,7,N,9999,60,201301,201712,0,0,0,0,None,0,01,201712,201712,0,0,1,0,0,NaN,0
1,F12Q40491228,2012Q4,786,201302,N,204301,0,1,P,115,999,119000,84,3.5,R,N,FRM,IL,SF,N,360,2,"JPMORGAN CHASE BANK, N.A.",N,None,Y,7,N,9999,162,201301,202606,0,0,0,0,None,0,NaN,NaN,NaN,0,0,0,0,0,NaN,0
2,F12Q40491233,2012Q4,796,201302,N,204301,0,1,P,60,999,131000,60,3.875,R,N,FRM,TN,SF,N,360,2,"JPMORGAN CHASE BANK, N.A.",N,None,Y,7,N,9999,54,201301,201706,0,0,0,0,None,0,01,201706,201706,0,0,1,0,0,NaN,0
3,F12Q40491235,2012Q4,768,201302,N,204301,0,1,P,63,999,143000,63,3.875,R,N,FRM,MI,SF,N,360,1,"JPMORGAN CHASE BANK, N.A.",N,None,Y,7,N,9999,36,201301,201512,0,0,0,0,None,0,01,201512,201512,0,0,1,0,0,NaN,0
4,F12Q40491237,2012Q4,796,201302,N,204301,0,1,P,63,999,146000,25,3.875,R,N,FRM,FL,PU,N,360,1,"JPMORGAN CHASE BANK, N.A.",N,None,Y,7,N,9999,78,201301,201906,0,0,0,0,None,0,01,201906,201906,0,0,1,0,0,NaN,0


EXPECTED ROWS: 592,713


In [12]:
# ============================================================
# CELL 7A: DEFINE LOAN-LEVEL AGGREGATION FUNCTION
# ============================================================

def build_loan_level_sql():

    return """
    WITH loan_aggregation AS (

        SELECT
            loan_identifier,

            COUNT(*) AS performance_months,
            MIN(period) AS first_observed_period,
            MAX(period) AS last_observed_period,

            MAX(
                CASE WHEN TRY_CAST(
                    TRIM(current_loan_delinquency_status) AS INTEGER
                ) >= 1 THEN 1 ELSE 0 END
            ) AS ever_30dpd,

            MAX(
                CASE WHEN TRY_CAST(
                    TRIM(current_loan_delinquency_status) AS INTEGER
                ) >= 2 THEN 1 ELSE 0 END
            ) AS ever_60dpd,

            MAX(
                CASE WHEN TRY_CAST(
                    TRIM(current_loan_delinquency_status) AS INTEGER
                ) >= 3 THEN 1 ELSE 0 END
            ) AS ever_90dpd,

            MAX(
                CASE WHEN TRIM(current_loan_delinquency_status) = 'RA'
                THEN 1 ELSE 0 END
            ) AS ever_reo,

            MIN(period) FILTER (
                WHERE TRY_CAST(
                    TRIM(current_loan_delinquency_status) AS INTEGER
                ) >= 3
            ) AS first_90dpd_period,

            MAX(
                TRY_CAST(
                    TRIM(current_loan_delinquency_status) AS INTEGER
                )
            ) AS max_numeric_delinquency_status,

            MAX(zero_balance_code) FILTER (
                WHERE NULLIF(TRIM(zero_balance_code), '') IS NOT NULL
            ) AS terminal_zero_balance_code,

            MAX(period) FILTER (
                WHERE NULLIF(TRIM(zero_balance_code), '') IS NOT NULL
            ) AS terminal_reporting_period,

            MAX(zero_balance_effective_date) FILTER (
                WHERE NULLIF(TRIM(zero_balance_code), '') IS NOT NULL
            ) AS zero_balance_effective_date,

            MAX(
                TRY_CAST(NULLIF(TRIM(actual_loss), '') AS DOUBLE)
            ) FILTER (
                WHERE zero_balance_code IN ('02', '03', '09', '15')
            ) AS actual_loss

        FROM read_parquet(?)
        GROUP BY loan_identifier
    )

    SELECT
        o.* EXCLUDE (loan_identifier),

        o.loan_identifier,
        ? AS vintage,

        a.performance_months,
        a.first_observed_period,
        a.last_observed_period,

        a.ever_30dpd,
        a.ever_60dpd,
        a.ever_90dpd,
        a.ever_reo,
        a.first_90dpd_period,
        a.max_numeric_delinquency_status,

        a.terminal_zero_balance_code,
        a.terminal_reporting_period,
        a.zero_balance_effective_date,

        CASE
            WHEN a.terminal_zero_balance_code IN ('02', '03', '09')
            THEN 1 ELSE 0
        END AS credit_related_termination,

        CASE
            WHEN a.terminal_zero_balance_code = '15'
            THEN 1 ELSE 0
        END AS whole_loan_sale,

        CASE
            WHEN a.terminal_zero_balance_code = '01'
            THEN 1 ELSE 0
        END AS voluntary_termination,

        CASE
            WHEN a.terminal_zero_balance_code = '96'
            THEN 1 ELSE 0
        END AS defect_termination,

        CASE
            WHEN a.terminal_zero_balance_code = '16'
            THEN 1 ELSE 0
        END AS reperforming_securitization,

        a.actual_loss,

        CASE
            WHEN a.actual_loss IS NOT NULL THEN 1 ELSE 0
        END AS actual_loss_available

    FROM read_parquet(?) o

    LEFT JOIN loan_aggregation a
        ON o.loan_identifier = a.loan_identifier
    """

In [13]:
# ============================================================
# CELL 7B: TEST REVISED AGGREGATION SQL
# ============================================================

test_vintage = "2012Q4"

test_perf = PERF_PARQUET_DIR / f"perf_{test_vintage}.parquet"
test_orig = ORIG_PARQUET_DIR / f"orig_{test_vintage}.parquet"

test_sql = build_loan_level_sql()

test_result = con.execute(
    test_sql,
    [
        str(test_perf),
        test_vintage,
        str(test_orig)
    ]
)

test_schema = con.execute(
    """
    DESCRIBE
    SELECT *
    FROM (
        """ + test_sql + """
    )
    LIMIT 0
    """,
    [
        str(test_perf),
        test_vintage,
        str(test_orig)
    ]
).df()

display(test_schema)

test_count = con.execute(
    """
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT loan_identifier) AS unique_loans
    FROM (
        """ + test_sql + """
    )
    """,
    [
        str(test_perf),
        test_vintage,
        str(test_orig)
    ]
).df()

display(test_count)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,column_name,column_type,null,key,default,extra
0,classic_fico,VARCHAR,YES,None,None,None
1,first_payment_date,VARCHAR,YES,None,None,None
2,first_time_homebuyer_indicator,VARCHAR,YES,None,None,None
3,maturity_date,VARCHAR,YES,None,None,None
4,metropolitan_statistical_area_msa_or_metropoli...,VARCHAR,YES,None,None,None
5,mortgage_insurance_percentage_mi,VARCHAR,YES,None,None,None
6,number_of_units,VARCHAR,YES,None,None,None
7,occupancy_status,VARCHAR,YES,None,None,None
8,original_combined_loan_to_value_cltv,VARCHAR,YES,None,None,None
9,original_debt_to_income_dti_ratio,VARCHAR,YES,None,None,None


,total_rows,unique_loans
0,592713,592713


In [14]:
# ============================================================
# CELL 7C.1: BATCH BUILD LOAN-LEVEL DATASET
# ============================================================

import os
import gc
import time
import pandas as pd
from pathlib import Path

manifest_path = EDA_DATA_DIR / "loan_level_manifest.csv"

manifest_records = []

overall_start = time.time()

for vintage in VINTAGES:

    print("\n" + "=" * 65)
    print(f"PROCESSING VINTAGE: {vintage}")
    print("=" * 65)

    perf_file = PERF_PARQUET_DIR / f"perf_{vintage}.parquet"
    orig_file = ORIG_PARQUET_DIR / f"orig_{vintage}.parquet"

    output_file = EDA_DATA_DIR / f"loan_level_{vintage}.parquet"
    temp_file = EDA_DATA_DIR / f"loan_level_{vintage}.tmp.parquet"

    start = time.time()
    status = "failed"
    error_message = None
    total_rows = None
    unique_loans = None

    db = None

    try:
        # 1. Check input files
        assert perf_file.exists(), f"Missing: {perf_file}"
        assert orig_file.exists(), f"Missing: {orig_file}"

        # 2. Create isolated DuckDB connection
        db = duckdb.connect()

        db.execute("SET memory_limit = '3GB'")
        db.execute("SET threads = 2")
        db.execute("SET preserve_insertion_order = false")

        temp_path = DUCKDB_TEMP_DIR.as_posix().replace("'", "''")
        db.execute(f"SET temp_directory = '{temp_path}'")

        # 3. Build loan-level temporary table
        sql = build_loan_level_sql()

        db.execute(
            f"""
            CREATE TEMP TABLE loan_level AS
            {sql}
            """,
            [
                str(perf_file),
                vintage,
                str(orig_file)
            ]
        )

        # 4. Validate in-memory temporary table
        validation = db.execute(
            """
            SELECT
                COUNT(*) AS total_rows,
                COUNT(DISTINCT loan_identifier) AS unique_loans,
                COUNT(*) FILTER (
                    WHERE loan_identifier IS NULL
                ) AS missing_ids
            FROM loan_level
            """
        ).fetchone()

        total_rows, unique_loans, missing_ids = validation

        if total_rows != unique_loans:
            raise ValueError(
                f"Duplicate loan IDs: {total_rows} rows, "
                f"{unique_loans} unique loans"
            )

        if missing_ids != 0:
            raise ValueError(f"Missing loan IDs: {missing_ids}")

        # 5. Write temporary Parquet
        temp_sql_path = str(temp_file).replace("'", "''")

        db.execute(
            f"""
            COPY loan_level
            TO '{temp_sql_path}'
            (FORMAT PARQUET, COMPRESSION ZSTD)
            """
        )

        # 6. Validate exported Parquet
        parquet_check = db.execute(
            """
            SELECT
                COUNT(*) AS total_rows,
                COUNT(DISTINCT loan_identifier) AS unique_loans
            FROM read_parquet(?)
            """,
            [str(temp_file)]
        ).fetchone()

        if parquet_check != (total_rows, unique_loans):
            raise ValueError(
                f"Parquet validation failed: {parquet_check}"
            )

        # 7. Promote temporary file to official output
        os.replace(temp_file, output_file)

        status = "completed"

        print(f"Rows: {total_rows:,}")
        print(f"Unique loans: {unique_loans:,}")
        print(f"Output: {output_file.name}")
        print(f"Time: {time.time() - start:.2f} seconds")

    except Exception as e:
        error_message = str(e)
        print("ERROR:", error_message)

        if temp_file.exists():
            temp_file.unlink()

    finally:
        if db is not None:
            db.close()

        gc.collect()

    # 8. Save manifest after every vintage
    record = {
        "vintage": vintage,
        "total_rows": total_rows,
        "unique_loans": unique_loans,
        "status": status,
        "execution_seconds": round(time.time() - start, 2),
        "error": error_message
    }

    manifest_records.append(record)

    current_manifest = pd.DataFrame(manifest_records)

    if manifest_path.exists():
        previous_manifest = pd.read_csv(manifest_path)
        current_manifest = pd.concat(
            [previous_manifest, current_manifest],
            ignore_index=True
        ).drop_duplicates(
            subset=["vintage"],
            keep="last"
        )

    current_manifest.to_csv(manifest_path, index=False)

print("\n" + "=" * 65)
print("BATCH PROCESSING FINISHED")
print("=" * 65)

print("Total execution time:",
      round(time.time() - overall_start, 2), "seconds")

display(pd.DataFrame(manifest_records))


PROCESSING VINTAGE: 2006Q1


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 298,574
Unique loans: 298,574
Output: loan_level_2006Q1.parquet
Time: 6.12 seconds

PROCESSING VINTAGE: 2006Q2


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 308,208
Unique loans: 308,208
Output: loan_level_2006Q2.parquet
Time: 7.41 seconds

PROCESSING VINTAGE: 2006Q3


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (57 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 592,713
Unique loans: 592,713
Output: loan_level_2012Q4.parquet
Time: 15.92 seconds

BATCH PROCESSING FINISHED
Total execution time: 265.57 seconds


,vintage,total_rows,unique_loans,status,execution_seconds,error
0,2006Q1,298574,298574,completed,6.7700,None
1,2006Q2,308208,308208,completed,7.6300,None
2,2006Q3,283975,283975,completed,6.4600,None
3,2006Q4,302787,302787,completed,7.9300,None
4,2007Q1,302331,302331,completed,7.1300,None
5,2007Q2,354055,354055,completed,7.8200,None
6,2007Q3,284980,284980,completed,6.9800,None
7,2007Q4,296826,296826,completed,5.9100,None
8,2008Q1,421635,421635,completed,9.3300,None
9,2008Q2,370982,370982,completed,7.5700,None


In [15]:
# ============================================================
# CELL 7C.2: FINAL BATCH AUDIT
# ============================================================

manifest = pd.read_csv(manifest_path)

loan_level_files = sorted(
    EDA_DATA_DIR.glob("loan_level_*.parquet")
)

loan_level_files = [
    f for f in loan_level_files
    if not f.name.endswith(".tmp.parquet")
]

print("BATCH COMPLETENESS")
print("-" * 55)

print("Expected vintages:", len(VINTAGES))
print("Completed vintages:",
      (manifest["status"] == "completed").sum())
print("Output Parquet files:", len(loan_level_files))

print("\nLOAN COUNT AUDIT")
print("-" * 55)

print("Total rows:", f"{manifest['total_rows'].sum():,.0f}")
print("Total unique loans:",
      f"{manifest['unique_loans'].sum():,.0f}")

print("\nFAILED VINTAGES")
display(manifest[manifest["status"] != "completed"])

assert (manifest["status"] == "completed").all()
assert len(loan_level_files) == 28
assert manifest["total_rows"].sum() == EXPECTED_LOANS
assert manifest["unique_loans"].sum() == EXPECTED_LOANS

print("\nALL BATCH ASSERTIONS PASSED.")

BATCH COMPLETENESS
-------------------------------------------------------
Expected vintages: 28
Completed vintages: 28
Output Parquet files: 28

LOAN COUNT AUDIT
-------------------------------------------------------
Total rows: 11,128,442
Total unique loans: 11,128,442

FAILED VINTAGES


,vintage,total_rows,unique_loans,status,execution_seconds,error



ALL BATCH ASSERTIONS PASSED.


In [16]:
# ============================================================
# CELL 8: LOAN-LEVEL DATA QUALITY AUDIT
# ============================================================

loan_files = [
    str(EDA_DATA_DIR / f"loan_level_{v}.parquet")
    for v in VINTAGES
]

# ------------------------------------------------------------
# 1. Overall data quality
# ------------------------------------------------------------

quality_summary = con.execute(
    """
    SELECT
        COUNT(*) AS total_loans,

        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(classic_fico), '') IS NULL
               OR TRIM(classic_fico) = '9999'
        ) AS missing_fico,

        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(original_debt_to_income_dti_ratio), '') IS NULL
               OR TRIM(original_debt_to_income_dti_ratio) = '999'
        ) AS missing_dti,

        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(original_loan_to_value_ltv), '') IS NULL
               OR TRIM(original_loan_to_value_ltv) = '999'
        ) AS missing_ltv,

        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(original_combined_loan_to_value_cltv), '') IS NULL
               OR TRIM(original_combined_loan_to_value_cltv) = '999'
        ) AS missing_cltv,

        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(number_of_units), '') IS NULL
               OR TRIM(number_of_units) = '99'
        ) AS missing_units,

        COUNT(*) FILTER (
            WHERE actual_loss IS NULL
        ) AS missing_actual_loss,

        COUNT(*) FILTER (
            WHERE terminal_zero_balance_code IS NULL
        ) AS loans_without_terminal_code

    FROM read_parquet(?)
    """,
    [loan_files]
).df()

display(quality_summary)


# ------------------------------------------------------------
# 2. Missing percentage
# ------------------------------------------------------------

missing_variables = [
    "missing_fico",
    "missing_dti",
    "missing_ltv",
    "missing_cltv",
    "missing_units",
    "missing_actual_loss",
    "loans_without_terminal_code"
]

missing_summary = quality_summary[
    missing_variables
].T.reset_index()

missing_summary.columns = [
    "variable",
    "missing_count"
]

missing_summary["missing_percentage"] = (
    missing_summary["missing_count"]
    / quality_summary.loc[0, "total_loans"]
    * 100
)

display(missing_summary)

,total_loans,missing_fico,missing_dti,missing_ltv,missing_cltv,missing_units,missing_actual_loss,loans_without_terminal_code
0,11128442,3067,1974757,461,477,2053,10821054,807161


,variable,missing_count,missing_percentage
0,missing_fico,3067,0.0276
1,missing_dti,1974757,17.7451
2,missing_ltv,461,0.0041
3,missing_cltv,477,0.0043
4,missing_units,2053,0.0184
5,missing_actual_loss,10821054,97.2378
6,loans_without_terminal_code,807161,7.2531


In [17]:
# ============================================================
# CELL 9: DTI MISSINGNESS BY HARP STATUS
# ============================================================

dti_harp_audit = con.execute(
    """
    SELECT
        harp_indicator,

        COUNT(*) AS total_loans,

        COUNT(*) FILTER (
            WHERE TRIM(original_debt_to_income_dti_ratio) = '999'
               OR NULLIF(
                    TRIM(original_debt_to_income_dti_ratio), ''
                  ) IS NULL
        ) AS missing_dti,

        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE TRIM(original_debt_to_income_dti_ratio) = '999'
                   OR NULLIF(
                        TRIM(original_debt_to_income_dti_ratio), ''
                      ) IS NULL
            ) / COUNT(*),
            2
        ) AS dti_missing_pct

    FROM read_parquet(?)

    GROUP BY harp_indicator
    ORDER BY harp_indicator
    """,
    [loan_files]
).df()

display(dti_harp_audit)

,harp_indicator,total_loans,missing_dti,dti_missing_pct
0,N,9241010,87325,0.9400
1,Y,1887432,1887432,100.0000


In [18]:
# ============================================================
# CELL 10A: FEATURE ENGINEERING - TEST 2012Q4
# ============================================================

test_vintage = "2012Q4"

input_file = (
    EDA_DATA_DIR / f"loan_level_{test_vintage}.parquet"
)

clean_test_file = (
    EDA_DATA_DIR / f"loan_level_clean_{test_vintage}.parquet"
)

clean_temp_file = (
    EDA_DATA_DIR / f"loan_level_clean_{test_vintage}.tmp.parquet"
)

start_time = time.time()

db = duckdb.connect()

try:
    db.execute("SET memory_limit = '3GB'")
    db.execute("SET threads = 2")
    db.execute("SET preserve_insertion_order = false")

    temp_path = DUCKDB_TEMP_DIR.as_posix().replace("'", "''")
    db.execute(f"SET temp_directory = '{temp_path}'")

    db.execute(
        """
        COPY (
            SELECT
                *,

                -- Borrower credit quality
                CASE
                    WHEN TRIM(classic_fico) = '9999'
                    THEN NULL
                    ELSE TRY_CAST(
                        NULLIF(TRIM(classic_fico), '') AS DOUBLE
                    )
                END AS classic_fico_num,

                -- Mortgage insurance
                CASE
                    WHEN TRIM(mortgage_insurance_percentage_mi) = '999'
                    THEN NULL
                    ELSE TRY_CAST(
                        NULLIF(
                            TRIM(mortgage_insurance_percentage_mi), ''
                        ) AS DOUBLE
                    )
                END AS mi_percentage_num,

                -- Property characteristics
                CASE
                    WHEN TRIM(number_of_units) = '99'
                    THEN NULL
                    ELSE TRY_CAST(
                        NULLIF(TRIM(number_of_units), '') AS DOUBLE
                    )
                END AS number_of_units_num,

                -- Leverage
                CASE
                    WHEN TRIM(original_combined_loan_to_value_cltv) = '999'
                    THEN NULL
                    ELSE TRY_CAST(
                        NULLIF(
                            TRIM(original_combined_loan_to_value_cltv), ''
                        ) AS DOUBLE
                    )
                END AS original_cltv_num,

                CASE
                    WHEN TRIM(original_debt_to_income_dti_ratio) = '999'
                    THEN NULL
                    ELSE TRY_CAST(
                        NULLIF(
                            TRIM(original_debt_to_income_dti_ratio), ''
                        ) AS DOUBLE
                    )
                END AS original_dti_num,

                CASE
                    WHEN TRIM(original_loan_to_value_ltv) = '999'
                    THEN NULL
                    ELSE TRY_CAST(
                        NULLIF(
                            TRIM(original_loan_to_value_ltv), ''
                        ) AS DOUBLE
                    )
                END AS original_ltv_num,

                -- Loan characteristics
                TRY_CAST(original_upb AS DOUBLE)
                    AS original_upb_num,

                TRY_CAST(original_interest_rate AS DOUBLE)
                    AS original_interest_rate_num,

                TRY_CAST(original_loan_term AS INTEGER)
                    AS original_loan_term_num,

                TRY_CAST(number_of_borrowers AS INTEGER)
                    AS number_of_borrowers_num

            FROM read_parquet(?)

        )
        TO ?
        (FORMAT PARQUET, COMPRESSION ZSTD)
        """,
        [str(input_file), str(clean_temp_file)]
    )

    # Validate output
    validation = db.execute(
        """
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT loan_identifier) AS unique_loans,
            COUNT(*) FILTER (
                WHERE classic_fico_num IS NOT NULL
            ) AS valid_fico,
            COUNT(*) FILTER (
                WHERE original_dti_num IS NOT NULL
            ) AS valid_dti
        FROM read_parquet(?)
        """,
        [str(clean_temp_file)]
    ).df()

    display(validation)

    assert validation.loc[0, "total_rows"] == 592713
    assert validation.loc[0, "unique_loans"] == 592713

    import os
    os.replace(clean_temp_file, clean_test_file)

    print("Cleaned analytical dataset created successfully.")
    print("Output:", clean_test_file)

finally:
    db.close()
    gc.collect()

print(f"Execution time: {time.time() - start_time:.2f} seconds")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_loans,valid_fico,valid_dti
0,592713,592713,592709,405290


Cleaned analytical dataset created successfully.
Output: E:\DS_project_backup\credit-risk-portfolio\processed\eda\loan_level\loan_level_clean_2012Q4.parquet
Execution time: 2.91 seconds


In [19]:
# ============================================================
# CELL 10B: INSPECT NUMERIC FEATURE DISTRIBUTIONS
# ============================================================

clean_sample = con.execute(
    """
    SELECT
        COUNT(*) AS total_loans,

        COUNT(classic_fico_num) AS valid_fico,
        MIN(classic_fico_num) AS min_fico,
        MAX(classic_fico_num) AS max_fico,
        AVG(classic_fico_num) AS avg_fico,

        COUNT(original_dti_num) AS valid_dti,
        MIN(original_dti_num) AS min_dti,
        MAX(original_dti_num) AS max_dti,
        AVG(original_dti_num) AS avg_dti,

        COUNT(original_ltv_num) AS valid_ltv,
        MIN(original_ltv_num) AS min_ltv,
        MAX(original_ltv_num) AS max_ltv,
        AVG(original_ltv_num) AS avg_ltv,

        COUNT(original_upb_num) AS valid_upb,
        MIN(original_upb_num) AS min_upb,
        MAX(original_upb_num) AS max_upb,
        AVG(original_upb_num) AS avg_upb

    FROM read_parquet(?)
    """,
    [str(clean_test_file)]
).df()

display(clean_sample)

,total_loans,valid_fico,min_fico,max_fico,avg_fico,valid_dti,min_dti,max_dti,avg_dti,valid_ltv,min_ltv,max_ltv,avg_ltv,valid_upb,min_upb,max_upb,avg_upb
0,592713,592709,435.0000,839.0000,758.5028,405290,1.0000,50.0000,30.7407,592705,1.0000,663.0000,74.5655,592713,"7,000.0000","1,200,000.0000","205,580.2201"


In [20]:
# ============================================================
# CELL 10C: NUMERIC RANGE VALIDATION - 2012Q4
# ============================================================

range_audit = con.execute(
    """
    SELECT
        COUNT(*) AS total_loans,

        -- FICO
        COUNT(*) FILTER (
            WHERE classic_fico_num < 300
               OR classic_fico_num > 850
        ) AS invalid_fico,

        MIN(classic_fico_num) AS min_fico,
        MAX(classic_fico_num) AS max_fico,

        -- DTI
        COUNT(*) FILTER (
            WHERE original_dti_num < 0
               OR original_dti_num > 65
        ) AS invalid_dti,

        MIN(original_dti_num) AS min_dti,
        MAX(original_dti_num) AS max_dti,

        -- LTV
        COUNT(*) FILTER (
            WHERE original_ltv_num < 6
               OR original_ltv_num > 105
        ) AS invalid_ltv,

        MIN(original_ltv_num) AS min_ltv,
        MAX(original_ltv_num) AS max_ltv,

        -- CLTV
        COUNT(*) FILTER (
            WHERE original_cltv_num < 6
               OR original_cltv_num > 200
        ) AS invalid_cltv,

        MIN(original_cltv_num) AS min_cltv,
        MAX(original_cltv_num) AS max_cltv

    FROM read_parquet(?)
    """,
    [str(clean_test_file)]
).df()

display(range_audit)

,total_loans,invalid_fico,min_fico,max_fico,invalid_dti,min_dti,max_dti,invalid_ltv,min_ltv,max_ltv,invalid_cltv,min_cltv,max_cltv
0,592713,0,435.0000,839.0000,0,1.0000,50.0000,48806,1.0000,663.0000,2523,1.0000,879.0000


In [21]:
# ============================================================
# CELL 10D: INSPECT ABNORMAL LTV OBSERVATIONS
# ============================================================

abnormal_ltv = con.execute(
    """
    SELECT
        loan_identifier,
        vintage,
        original_loan_to_value_ltv,
        original_ltv_num,
        original_combined_loan_to_value_cltv,
        original_cltv_num,
        classic_fico_num,
        original_dti_num,
        original_upb_num,
        harp_indicator,
        special_eligibility_program

    FROM read_parquet(?)

    WHERE original_ltv_num < 6
       OR original_ltv_num > 105
       OR original_cltv_num < 6
       OR original_cltv_num > 200

    ORDER BY original_ltv_num DESC

    LIMIT 30
    """,
    [str(clean_test_file)]
).df()

print("Number of abnormal observations:", len(abnormal_ltv))
display(abnormal_ltv)

Number of abnormal observations: 30


,loan_identifier,vintage,original_loan_to_value_ltv,original_ltv_num,original_combined_loan_to_value_cltv,original_cltv_num,classic_fico_num,original_dti_num,original_upb_num,harp_indicator,special_eligibility_program
0,F12Q40199517,2012Q4,663,663.0000,663,663.0000,710.0000,NaN,"126,000.0000",Y,None
1,F12Q40586299,2012Q4,560,560.0000,560,560.0000,751.0000,NaN,"307,000.0000",Y,None
2,F12Q40264749,2012Q4,556,556.0000,556,556.0000,810.0000,NaN,"100,000.0000",Y,None
3,F12Q40365253,2012Q4,546,546.0000,546,546.0000,709.0000,NaN,"153,000.0000",Y,None
4,F12Q40488192,2012Q4,540,540.0000,540,540.0000,577.0000,NaN,"49,000.0000",Y,None
5,F12Q40451036,2012Q4,525,525.0000,525,525.0000,749.0000,NaN,"79,000.0000",Y,None
6,F12Q40451127,2012Q4,522,522.0000,522,522.0000,627.0000,NaN,"91,000.0000",Y,None
7,F12Q40395800,2012Q4,509,509.0000,509,509.0000,772.0000,NaN,"102,000.0000",Y,None
8,F12Q40216320,2012Q4,508,508.0000,645,645.0000,787.0000,NaN,"75,000.0000",Y,None
9,F12Q40078242,2012Q4,491,491.0000,491,491.0000,813.0000,NaN,"137,000.0000",Y,None


In [22]:
# ============================================================
# CELL 10E: LTV / CLTV DISTRIBUTION BY HARP STATUS
# ============================================================

harp_ltv_audit = con.execute(
    """
    SELECT
        harp_indicator,

        COUNT(*) AS total_loans,

        COUNT(*) FILTER (
            WHERE original_ltv_num > 105
        ) AS ltv_above_105,

        COUNT(*) FILTER (
            WHERE original_cltv_num > 200
        ) AS cltv_above_200,

        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE original_ltv_num > 105
            ) / COUNT(*),
            3
        ) AS pct_ltv_above_105,

        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE original_cltv_num > 200
            ) / COUNT(*),
            3
        ) AS pct_cltv_above_200,

        MIN(original_ltv_num) AS min_ltv,
        MEDIAN(original_ltv_num) AS median_ltv,
        MAX(original_ltv_num) AS max_ltv,

        MIN(original_cltv_num) AS min_cltv,
        MEDIAN(original_cltv_num) AS median_cltv,
        MAX(original_cltv_num) AS max_cltv

    FROM read_parquet(?)

    GROUP BY harp_indicator
    ORDER BY harp_indicator
    """,
    [str(clean_test_file)]
).df()

display(harp_ltv_audit)

,harp_indicator,total_loans,ltv_above_105,cltv_above_200,pct_ltv_above_105,pct_cltv_above_200,min_ltv,median_ltv,max_ltv,min_cltv,median_cltv,max_cltv
0,N,405305,0,0,0.0000,0.0000,6.0000,73.0000,100.0000,6.0000,74.0000,135.0000
1,Y,187408,48776,2504,26.0270,1.3360,1.0000,86.0000,663.0000,1.0000,92.0000,879.0000


In [23]:
# ============================================================
# CELL 10F: HARP HIGH-LTV DISTRIBUTION
# ============================================================

harp_high_ltv = con.execute(
    """
    SELECT
        CASE
            WHEN original_ltv_num <= 80 THEN 'LTV <= 80'
            WHEN original_ltv_num <= 100 THEN '80 < LTV <= 100'
            WHEN original_ltv_num <= 105 THEN '100 < LTV <= 105'
            WHEN original_ltv_num <= 125 THEN '105 < LTV <= 125'
            WHEN original_ltv_num <= 150 THEN '125 < LTV <= 150'
            WHEN original_ltv_num <= 200 THEN '150 < LTV <= 200'
            ELSE 'LTV > 200'
        END AS ltv_group,

        COUNT(*) AS loan_count,

        ROUND(
            100.0 * COUNT(*) /
            SUM(COUNT(*)) OVER (),
            2
        ) AS portfolio_percentage

    FROM read_parquet(?)

    WHERE harp_indicator = 'Y'
      AND original_ltv_num IS NOT NULL

    GROUP BY ltv_group

    ORDER BY
        MIN(original_ltv_num)
    """
    ,
    [str(clean_test_file)]
).df()

display(harp_high_ltv)

,ltv_group,loan_count,portfolio_percentage
0,LTV <= 80,78001,41.6200
1,80 < LTV <= 100,50178,26.7700
2,100 < LTV <= 105,10453,5.5800
3,105 < LTV <= 125,25078,13.3800
4,125 < LTV <= 150,14006,7.4700
5,150 < LTV <= 200,7950,4.2400
6,LTV > 200,1742,0.9300


In [24]:
# ============================================================
# CELL 10G: REPRINT HARP LTV AUDIT
# ============================================================

harp_ltv_audit = con.execute(
    """
    SELECT
        harp_indicator,
        COUNT(*) AS total_loans,

        COUNT(*) FILTER (
            WHERE original_ltv_num > 105
        ) AS ltv_above_105,

        COUNT(*) FILTER (
            WHERE original_cltv_num > 200
        ) AS cltv_above_200,

        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE original_ltv_num > 105
            ) / COUNT(*), 3
        ) AS pct_ltv_above_105,

        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE original_cltv_num > 200
            ) / COUNT(*), 3
        ) AS pct_cltv_above_200,

        MIN(original_ltv_num) AS min_ltv,
        MEDIAN(original_ltv_num) AS median_ltv,
        MAX(original_ltv_num) AS max_ltv,

        MIN(original_cltv_num) AS min_cltv,
        MEDIAN(original_cltv_num) AS median_cltv,
        MAX(original_cltv_num) AS max_cltv

    FROM read_parquet(?)
    GROUP BY harp_indicator
    ORDER BY harp_indicator
    """,
    [str(clean_test_file)]
).df()

print(harp_ltv_audit.to_string(index=False))

harp_indicator  total_loans  ltv_above_105  cltv_above_200  pct_ltv_above_105  pct_cltv_above_200  min_ltv  median_ltv  max_ltv  min_cltv  median_cltv  max_cltv
             N       405305              0               0             0.0000              0.0000   6.0000     73.0000 100.0000    6.0000      74.0000  135.0000
             Y       187408          48776            2504            26.0270              1.3360   1.0000     86.0000 663.0000    1.0000      92.0000  879.0000


In [25]:
# ============================================================
# CELL 10H: REPRINT HARP LTV DISTRIBUTION
# ============================================================

harp_high_ltv = con.execute(
    """
    SELECT
        CASE
            WHEN original_ltv_num <= 80 THEN 'LTV <= 80'
            WHEN original_ltv_num <= 100 THEN '80 < LTV <= 100'
            WHEN original_ltv_num <= 105 THEN '100 < LTV <= 105'
            WHEN original_ltv_num <= 125 THEN '105 < LTV <= 125'
            WHEN original_ltv_num <= 150 THEN '125 < LTV <= 150'
            WHEN original_ltv_num <= 200 THEN '150 < LTV <= 200'
            ELSE 'LTV > 200'
        END AS ltv_group,

        COUNT(*) AS loan_count,

        ROUND(
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
            2
        ) AS portfolio_percentage

    FROM read_parquet(?)
    WHERE harp_indicator = 'Y'
      AND original_ltv_num IS NOT NULL

    GROUP BY ltv_group
    ORDER BY MIN(original_ltv_num)
    """,
    [str(clean_test_file)]
).df()

print(harp_high_ltv.to_string(index=False))

       ltv_group  loan_count  portfolio_percentage
       LTV <= 80       78001               41.6200
 80 < LTV <= 100       50178               26.7700
100 < LTV <= 105       10453                5.5800
105 < LTV <= 125       25078               13.3800
125 < LTV <= 150       14006                7.4700
150 < LTV <= 200        7950                4.2400
       LTV > 200        1742                0.9300


In [26]:
# ============================================================
# CELL 10I: DIAGNOSE HARP - LTV INCONSISTENCY
# ============================================================

# 1. Cross-tabulate HARP status and LTV/CLTV conditions

harp_ltv_crosstab = con.execute(
    """
    SELECT
        harp_indicator,

        COUNT(*) AS total_loans,

        COUNT(*) FILTER (
            WHERE original_ltv_num <= 80
        ) AS ltv_le_80,

        COUNT(*) FILTER (
            WHERE original_ltv_num > 80
        ) AS ltv_gt_80,

        COUNT(*) FILTER (
            WHERE original_cltv_num <= 80
        ) AS cltv_le_80,

        COUNT(*) FILTER (
            WHERE original_cltv_num > 80
        ) AS cltv_gt_80,

        COUNT(*) FILTER (
            WHERE original_ltv_num <= 80
              AND original_cltv_num > 80
        ) AS ltv_le_80_cltv_gt_80,

        COUNT(*) FILTER (
            WHERE original_ltv_num <= 80
              AND original_cltv_num <= 80
        ) AS both_le_80

    FROM read_parquet(?)
    GROUP BY harp_indicator
    ORDER BY harp_indicator
    """
    ,
    [str(clean_test_file)]
).df()

print("HARP - LTV/CLTV CROSS-TABULATION")
print(harp_ltv_crosstab.to_string(index=False))


# 2. Inspect raw value frequencies for HARP loans

harp_ltv_values = con.execute(
    """
    SELECT
        original_loan_to_value_ltv AS raw_ltv,
        original_ltv_num AS numeric_ltv,
        original_combined_loan_to_value_cltv AS raw_cltv,
        original_cltv_num AS numeric_cltv,
        COUNT(*) AS loan_count

    FROM read_parquet(?)

    WHERE harp_indicator = 'Y'

    GROUP BY
        original_loan_to_value_ltv,
        original_ltv_num,
        original_combined_loan_to_value_cltv,
        original_cltv_num

    ORDER BY loan_count DESC

    LIMIT 30
    """,
    [str(clean_test_file)]
).df()

print("\nMOST FREQUENT HARP LTV/CLTV VALUES")
print(harp_ltv_values.to_string(index=False))


# 3. Inspect selected HARP loans with LTV <= 80

harp_low_ltv_examples = con.execute(
    """
    SELECT
        loan_identifier,
        classic_fico,
        original_loan_to_value_ltv,
        original_combined_loan_to_value_cltv,
        original_debt_to_income_dti_ratio,
        harp_indicator,
        pre_harp_loan_sequence_number,
        special_eligibility_program,
        property_valuation_method,
        original_upb

    FROM read_parquet(?)

    WHERE harp_indicator = 'Y'
      AND original_ltv_num <= 80

    ORDER BY original_ltv_num

    LIMIT 30
    """,
    [str(clean_test_file)]
).df()

print("\nHARP LOANS WITH LTV <= 80")
print(harp_low_ltv_examples.to_string(index=False))

HARP - LTV/CLTV CROSS-TABULATION
harp_indicator  total_loans  ltv_le_80  ltv_gt_80  cltv_le_80  cltv_gt_80  ltv_le_80_cltv_gt_80  both_le_80
             N       405305     348980      56317      337617       67680                 11363      337617
             Y       187408      78001     109407       66201      121206                 11799       66201

MOST FREQUENT HARP LTV/CLTV VALUES
raw_ltv  numeric_ltv raw_cltv  numeric_cltv  loan_count
    105     105.0000      105      105.0000        1921
     89      89.0000       89       89.0000        1898
     88      88.0000       88       88.0000        1888
     86      86.0000       86       86.0000        1853
     90      90.0000       90       90.0000        1852
     87      87.0000       87       87.0000        1845
     95      95.0000       95       95.0000        1831
     80      80.0000       80       80.0000        1807
     94      94.0000       94       94.0000        1774
     85      85.0000       85       85.0000    

In [27]:
# CELL 10J — HARP OLTV/CLTV validity diagnostics

# 1. Break down HARP loans by OLTV validity bands
harp_validity = con.execute("""
SELECT
    COUNT(*) AS total_harp_loans,

    COUNT(*) FILTER (
        WHERE original_ltv_num < 6
    ) AS ltv_below_6,

    COUNT(*) FILTER (
        WHERE original_ltv_num BETWEEN 6 AND 80
    ) AS ltv_6_to_80,

    COUNT(*) FILTER (
        WHERE original_ltv_num BETWEEN 81 AND 105
    ) AS ltv_81_to_105,

    COUNT(*) FILTER (
        WHERE original_ltv_num > 105
    ) AS ltv_above_105,

    COUNT(*) FILTER (
        WHERE original_ltv_num IS NULL
    ) AS ltv_missing,

    COUNT(*) FILTER (
        WHERE original_cltv_num < 6
    ) AS cltv_below_6,

    COUNT(*) FILTER (
        WHERE original_cltv_num BETWEEN 6 AND 80
    ) AS cltv_6_to_80,

    COUNT(*) FILTER (
        WHERE original_cltv_num BETWEEN 81 AND 200
    ) AS cltv_81_to_200,

    COUNT(*) FILTER (
        WHERE original_cltv_num > 200
    ) AS cltv_above_200,

    COUNT(*) FILTER (
        WHERE original_cltv_num IS NULL
    ) AS cltv_missing

FROM read_parquet(?)
WHERE TRIM(harp_indicator) = 'Y'
""", [str(clean_test_file)]).df()

print("HARP LTV/CLTV VALIDITY BREAKDOWN")
print(harp_validity.to_string(index=False))


# 2. Check whether invalid low values are concentrated
# in particular vintages
harp_by_vintage = con.execute("""
SELECT
    vintage,
    COUNT(*) AS harp_loans,

    COUNT(*) FILTER (
        WHERE original_ltv_num < 6
    ) AS ltv_below_6,

    COUNT(*) FILTER (
        WHERE original_ltv_num BETWEEN 6 AND 80
    ) AS ltv_6_to_80,

    COUNT(*) FILTER (
        WHERE original_ltv_num > 105
    ) AS ltv_above_105,

    MIN(original_ltv_num) AS min_ltv,
    MEDIAN(original_ltv_num) AS median_ltv,
    MAX(original_ltv_num) AS max_ltv

FROM read_parquet(?)
WHERE TRIM(harp_indicator) = 'Y'
GROUP BY vintage
ORDER BY vintage
""", [str(clean_test_file)]).df()

print("\nHARP LTV BY VINTAGE — 2012Q4 TEST FILE")
print(harp_by_vintage.to_string(index=False))


# 3. Inspect the frequency of suspicious raw LTV values
suspicious_values = con.execute("""
SELECT
    original_loan_to_value_ltv AS raw_ltv,
    original_combined_loan_to_value_cltv AS raw_cltv,
    COUNT(*) AS loan_count
FROM read_parquet(?)
WHERE TRIM(harp_indicator) = 'Y'
  AND TRY_CAST(
      TRIM(original_loan_to_value_ltv) AS DOUBLE
  ) < 6
GROUP BY 1, 2
ORDER BY loan_count DESC
LIMIT 30
""", [str(clean_test_file)]).df()

print("\nHARP RAW VALUES BELOW 6")
print(suspicious_values.to_string(index=False))

HARP LTV/CLTV VALIDITY BREAKDOWN
 total_harp_loans  ltv_below_6  ltv_6_to_80  ltv_81_to_105  ltv_above_105  ltv_missing  cltv_below_6  cltv_6_to_80  cltv_81_to_200  cltv_above_200  cltv_missing
           187408           30        77971          60631          48776            0            19         66182          118702            2504             1

HARP LTV BY VINTAGE — 2012Q4 TEST FILE
vintage  harp_loans  ltv_below_6  ltv_6_to_80  ltv_above_105  min_ltv  median_ltv  max_ltv
 2012Q4      187408           30        77971          48776   1.0000     86.0000 663.0000

HARP RAW VALUES BELOW 6
raw_ltv raw_cltv  loan_count
      5        5          10
      2        2           4
      1        2           2
      4        4           2
      5       10           2
      1        1           1
      5       16           1
      5       15           1
      5        7           1
      5       19           1
      4        7           1
      4       10           1
      5        8     

In [28]:
# CELL 10K — Locate raw 2012Q4 origination file

raw_roots = [
    Path(r"D:\DS Project\credit-portfolio-risk-modeling\dataset"),
    Path(r"E:\DS_project_backup\credit-risk-portfolio\data")
]

for root in raw_roots:
    print(f"\nSearching: {root}")
    if root.exists():
        for f in sorted(root.rglob("*")):
            if f.is_file() and "2012Q4" in f.name.upper():
                print(f.name, "|", f.stat().st_size, "bytes")
    else:
        print("Directory does not exist.")


Searching: D:\DS Project\credit-portfolio-risk-modeling\dataset
historical_data_2012Q4.zip | 873576527 bytes

Searching: E:\DS_project_backup\credit-risk-portfolio\data
orig_2012Q4.txt | 75802743 bytes
perf_2012Q4.txt | 6386578398 bytes


In [30]:
# CELL 10L.1 — Inspect the original 31-column mapping

mapping_file = AUDIT_DIR / "origination_mapping.csv"

orig_mapping_check = pd.read_csv(mapping_file)

print("Mapping shape:", orig_mapping_check.shape)
print("\nMapping columns:")
print(orig_mapping_check.columns.tolist())

print("\nFull Origination mapping:")
print(orig_mapping_check.to_string(index=False))

Mapping shape: (31, 7)

Mapping columns:
['dataset', 'Field Position', 'Attribute Name', 'Data Type & Format', 'Max Length', 'standard_name', 'data_category']

Full Origination mapping:
    dataset  Field Position                                               Attribute Name           Data Type & Format  Max Length                                              standard_name data_category
origination               1                                                Classic FICO®                      Numeric           4                                               classic_fico       numeric
origination               2                                           First Payment Date                         Date           6                                         first_payment_date          date
origination               3                               First Time Homebuyer Indicator                        Alpha           1                             first_time_homebuyer_indicator         alpha
or

In [31]:
# CELL 10L.2 — Read raw Origination TXT using the verified mapping

# Load mapping and preserve the original field order
orig_mapping = pd.read_csv(
    AUDIT_DIR / "origination_mapping.csv"
).sort_values("Field Position")

# Build ordered column definitions
orig_columns = orig_mapping["standard_name"].tolist()

assert len(orig_columns) == 31
assert orig_columns[8] == "original_combined_loan_to_value_cltv"
assert orig_columns[11] == "original_loan_to_value_ltv"
assert orig_columns[19] == "loan_identifier"
assert orig_columns[25] == "pre_harp_loan_sequence_number"
assert orig_columns[27] == "harp_indicator"

# Build DuckDB column dictionary
column_definitions = ", ".join(
    f"'{col}': 'VARCHAR'"
    for col in orig_columns
)

raw_orig_file = Path(
    r"E:\DS_project_backup\credit-risk-portfolio\data\orig_2012Q4.txt"
)

# Read a small sample first
raw_sample = con.execute(f"""
    SELECT
        loan_identifier,
        original_loan_to_value_ltv,
        original_combined_loan_to_value_cltv,
        harp_indicator,
        pre_harp_loan_sequence_number
    FROM read_csv(
        ?,
        delim='|',
        header=false,
        auto_detect=false,
        columns={{{column_definitions}}},
        quote='',
        escape=''
    )
    LIMIT 10
""", [str(raw_orig_file)]).df()

print("RAW TXT SAMPLE")
print(raw_sample.to_string(index=False))

RAW TXT SAMPLE
loan_identifier original_loan_to_value_ltv original_combined_loan_to_value_cltv harp_indicator pre_harp_loan_sequence_number
   F12Q40000001                         71                                   71              N                           NaN
   F12Q40000002                         71                                   71              N                           NaN
   F12Q40000003                         80                                   80              N                           NaN
   F12Q40000004                         95                                   95              N                           NaN
   F12Q40000005                        106                                  106              Y                  F09Q20008428
   F12Q40000006                         66                                   95              N                           NaN
   F12Q40000007                         76                                   86              Y                

In [32]:
# CELL 10L.3 — Raw TXT vs Parquet reconciliation

raw_relation = f"""
read_csv(
    ?,
    delim='|',
    header=false,
    auto_detect=false,
    columns={{{column_definitions}}},
    quote='',
    escape=''
)
"""

# 1. Row count and unique loan identifiers
raw_stats = con.execute(f"""
    SELECT
        COUNT(*) AS raw_rows,
        COUNT(DISTINCT loan_identifier) AS unique_loan_ids,
        COUNT(*) FILTER (
            WHERE TRIM(harp_indicator) = 'Y'
        ) AS raw_harp_loans
    FROM {raw_relation}
""", [str(raw_orig_file)]).df()

parquet_stats = con.execute("""
    SELECT
        COUNT(*) AS parquet_rows,
        COUNT(DISTINCT loan_identifier) AS unique_loan_ids,
        COUNT(*) FILTER (
            WHERE TRIM(harp_indicator) = 'Y'
        ) AS parquet_harp_loans
    FROM read_parquet(?)
""", [str(orig_parquet_file)]).df()

print("RAW TXT STATISTICS")
print(raw_stats.to_string(index=False))

print("\nPARQUET STATISTICS")
print(parquet_stats.to_string(index=False))


# 2. Compare HARP loan-level values
comparison = con.execute(f"""
    WITH raw AS (
        SELECT
            loan_identifier,
            original_loan_to_value_ltv AS raw_ltv,
            original_combined_loan_to_value_cltv AS raw_cltv,
            harp_indicator AS raw_harp
        FROM {raw_relation}
    ),
    parquet AS (
        SELECT
            loan_identifier,
            original_loan_to_value_ltv AS parquet_ltv,
            original_combined_loan_to_value_cltv AS parquet_cltv,
            harp_indicator AS parquet_harp
        FROM read_parquet(?)
    )
    SELECT
        COUNT(*) AS matched_loans,

        COUNT(*) FILTER (
            WHERE TRIM(raw_harp) = 'Y'
        ) AS matched_harp_loans,

        COUNT(*) FILTER (
            WHERE TRIM(raw_harp) = 'Y'
              AND TRY_CAST(TRIM(raw_ltv) AS DOUBLE) <= 80
        ) AS raw_harp_ltv_le_80,

        COUNT(*) FILTER (
            WHERE TRIM(raw_harp) = 'Y'
              AND raw_ltv IS DISTINCT FROM parquet_ltv
        ) AS ltv_mismatches,

        COUNT(*) FILTER (
            WHERE TRIM(raw_harp) = 'Y'
              AND raw_cltv IS DISTINCT FROM parquet_cltv
        ) AS cltv_mismatches,

        COUNT(*) FILTER (
            WHERE raw_harp IS DISTINCT FROM parquet_harp
        ) AS harp_mismatches

    FROM raw
    INNER JOIN parquet USING (loan_identifier)
""", [str(raw_orig_file), str(orig_parquet_file)]).df()

print("\nRAW TXT vs PARQUET COMPARISON")
print(comparison.to_string(index=False))

RAW TXT STATISTICS
 raw_rows  unique_loan_ids  raw_harp_loans
   592713           592713          187408

PARQUET STATISTICS
 parquet_rows  unique_loan_ids  parquet_harp_loans
       592713           592713              187408

RAW TXT vs PARQUET COMPARISON
 matched_loans  matched_harp_loans  raw_harp_ltv_le_80  ltv_mismatches  cltv_mismatches  harp_mismatches
        592713              187408               78001               0                0                0


In [33]:
# CELL 10M — Investigate pre-HARP loan characteristics

# All 28 Origination Parquet files
orig_files_for_join = [
    str(f) for f in sorted(ORIG_PARQUET_DIR.glob("orig_*.parquet"))
]

assert len(orig_files_for_join) == 28

pre_harp_analysis = con.execute("""
WITH historical_loans AS (
    SELECT
        loan_identifier,
        TRY_CAST(
            NULLIF(TRIM(original_loan_to_value_ltv), '')
            AS DOUBLE
        ) AS previous_ltv,
        TRY_CAST(
            NULLIF(TRIM(original_combined_loan_to_value_cltv), '')
            AS DOUBLE
        ) AS previous_cltv,
        original_upb AS previous_upb
    FROM read_parquet(?)
),

current_harp AS (
    SELECT
        loan_identifier AS current_loan_id,
        pre_harp_loan_sequence_number,
        original_loan_to_value_ltv AS current_ltv_raw,
        original_combined_loan_to_value_cltv AS current_cltv_raw,
        original_ltv_num AS current_ltv,
        original_cltv_num AS current_cltv
    FROM read_parquet(?)
    WHERE TRIM(harp_indicator) = 'Y'
)

SELECT
    COUNT(*) AS total_harp_loans,

    COUNT(*) FILTER (
        WHERE current_ltv <= 80
    ) AS current_ltv_le_80,

    COUNT(*) FILTER (
        WHERE current_ltv <= 80
          AND NULLIF(TRIM(pre_harp_loan_sequence_number), '') IS NOT NULL
    ) AS ltv_le_80_with_previous_id,

    COUNT(*) FILTER (
        WHERE current_ltv <= 80
          AND h.loan_identifier IS NOT NULL
    ) AS ltv_le_80_previous_loan_matched,

    COUNT(*) FILTER (
        WHERE current_ltv <= 80
          AND h.previous_ltv > 80
    ) AS current_le_80_previous_gt_80,

    COUNT(*) FILTER (
        WHERE current_ltv <= 80
          AND h.previous_ltv <= 80
    ) AS current_le_80_previous_le_80,

    COUNT(*) FILTER (
        WHERE current_ltv <= 80
          AND h.previous_ltv IS NULL
    ) AS current_le_80_previous_ltv_missing,

    MEDIAN(h.previous_ltv) FILTER (
        WHERE current_ltv <= 80
    ) AS median_previous_ltv,

    MEDIAN(current_ltv) FILTER (
        WHERE current_ltv <= 80
    ) AS median_current_ltv

FROM current_harp c
LEFT JOIN historical_loans h
    ON TRIM(c.pre_harp_loan_sequence_number) = h.loan_identifier
""", [orig_files_for_join, str(clean_test_file)]).df()

print("PRE-HARP LOAN MATCHING RESULTS")
print(pre_harp_analysis.to_string(index=False))

PRE-HARP LOAN MATCHING RESULTS
 total_harp_loans  current_ltv_le_80  ltv_le_80_with_previous_id  ltv_le_80_previous_loan_matched  current_le_80_previous_gt_80  current_le_80_previous_le_80  current_le_80_previous_ltv_missing  median_previous_ltv  median_current_ltv
           187408              78001                       78001                            25890                          1337                         24553                               52111              65.0000             60.0000


In [34]:
# CELL 10N — Analyze unmatched pre-HARP loan identifiers

pre_harp_matching_detail = con.execute("""
WITH historical_loans AS (
    SELECT
        loan_identifier,
        TRY_CAST(
            NULLIF(TRIM(original_loan_to_value_ltv), '')
            AS DOUBLE
        ) AS previous_ltv
    FROM read_parquet(?)
),

current_harp AS (
    SELECT
        loan_identifier AS current_loan_id,
        TRIM(pre_harp_loan_sequence_number) AS previous_loan_id,
        original_ltv_num AS current_ltv
    FROM read_parquet(?)
    WHERE TRIM(harp_indicator) = 'Y'
      AND original_ltv_num <= 80
),

matched AS (
    SELECT
        c.*,
        h.loan_identifier AS matched_previous_id,
        h.previous_ltv
    FROM current_harp c
    LEFT JOIN historical_loans h
        ON c.previous_loan_id = h.loan_identifier
)

SELECT
    SUBSTR(previous_loan_id, 1, 6) AS previous_vintage_prefix,

    COUNT(*) AS total_current_harp_loans,

    COUNT(*) FILTER (
        WHERE matched_previous_id IS NOT NULL
    ) AS matched_loans,

    COUNT(*) FILTER (
        WHERE matched_previous_id IS NULL
    ) AS unmatched_loans,

    MEDIAN(previous_ltv) AS median_previous_ltv,
    MEDIAN(current_ltv) AS median_current_ltv

FROM matched
GROUP BY 1
ORDER BY 1
""", [orig_files_for_join, str(clean_test_file)]).df()

print("PRE-HARP LOAN MATCHING BY VINTAGE PREFIX")
print(pre_harp_matching_detail.to_string(index=False))

PRE-HARP LOAN MATCHING BY VINTAGE PREFIX
previous_vintage_prefix  total_current_harp_loans  matched_loans  unmatched_loans  median_previous_ltv  median_current_ltv
                 A00Q20                         1              0                1                  NaN             55.0000
                 A01Q30                         2              0                2                  NaN             66.0000
                 A01Q40                         2              0                2                  NaN             54.5000
                 A02Q10                         7              0                7                  NaN             60.0000
                 A02Q20                         1              0                1                  NaN             80.0000
                 A02Q30                        10              0               10                  NaN             56.5000
                 A02Q40                        14              0               14                 

In [35]:
# CELL 10O — Investigate unmatched pre-HARP IDs within dataset coverage

unmatched_in_scope = con.execute("""
WITH historical_loans AS (
    SELECT DISTINCT
        TRIM(loan_identifier) AS loan_identifier
    FROM read_parquet(?)
),

current_harp AS (
    SELECT
        loan_identifier AS current_loan_id,
        TRIM(pre_harp_loan_sequence_number) AS previous_loan_id,
        original_ltv_num AS current_ltv
    FROM read_parquet(?)
    WHERE TRIM(harp_indicator) = 'Y'
      AND original_ltv_num <= 80
),

matched AS (
    SELECT
        c.*,
        h.loan_identifier AS matched_id
    FROM current_harp c
    LEFT JOIN historical_loans h
        ON c.previous_loan_id = h.loan_identifier
)

SELECT
    SUBSTR(previous_loan_id, 1, 6) AS previous_vintage_prefix,
    COUNT(*) AS total_loans,
    COUNT(*) FILTER (
        WHERE matched_id IS NOT NULL
    ) AS matched_loans,
    COUNT(*) FILTER (
        WHERE matched_id IS NULL
    ) AS unmatched_loans
FROM matched
WHERE SUBSTR(previous_loan_id, 1, 3) = 'F'
  AND TRY_CAST(SUBSTR(previous_loan_id, 2, 2) AS INTEGER)
      BETWEEN 6 AND 9
GROUP BY 1
ORDER BY 1
""", [orig_files_for_join, str(clean_test_file)]).df()

print("UNMATCHED PRE-HARP IDS WITHIN 2006–2009")
print(unmatched_in_scope.to_string(index=False))


# Inspect 30 unmatched examples
unmatched_examples = con.execute("""
WITH historical_loans AS (
    SELECT DISTINCT TRIM(loan_identifier) AS loan_identifier
    FROM read_parquet(?)
),

current_harp AS (
    SELECT
        loan_identifier AS current_loan_id,
        TRIM(pre_harp_loan_sequence_number) AS previous_loan_id,
        original_ltv_num AS current_ltv,
        original_cltv_num AS current_cltv
    FROM read_parquet(?)
    WHERE TRIM(harp_indicator) = 'Y'
      AND original_ltv_num <= 80
)

SELECT
    c.current_loan_id,
    c.previous_loan_id,
    c.current_ltv,
    c.current_cltv
FROM current_harp c
LEFT JOIN historical_loans h
    ON c.previous_loan_id = h.loan_identifier
WHERE h.loan_identifier IS NULL
  AND SUBSTR(c.previous_loan_id, 1, 3) = 'F'
  AND TRY_CAST(SUBSTR(c.previous_loan_id, 2, 2) AS INTEGER)
      BETWEEN 6 AND 9
LIMIT 30
""", [orig_files_for_join, str(clean_test_file)]).df()

print("\nUNMATCHED LOAN ID EXAMPLES")
print(unmatched_examples.to_string(index=False))

UNMATCHED PRE-HARP IDS WITHIN 2006–2009
Empty DataFrame
Columns: [previous_vintage_prefix, total_loans, matched_loans, unmatched_loans]
Index: []

UNMATCHED LOAN ID EXAMPLES
Empty DataFrame
Columns: [current_loan_id, previous_loan_id, current_ltv, current_cltv]
Index: []


In [38]:
# CELL 10P — Create cleaned test file with LTV/CLTV quality flags

source_test_file = EDA_DATA_DIR / "loan_level_2012Q4.parquet"
clean_test_v2 = EDA_DATA_DIR / "loan_level_clean_2012Q4_test_v2.parquet"

# Safely format file paths for SQL
source_path_sql = str(source_test_file).replace("\\", "/").replace("'", "''")
output_path_sql = str(clean_test_v2).replace("\\", "/").replace("'", "''")

# 1. Create cleaned view
con.execute(f"""
CREATE OR REPLACE TEMP VIEW cleaned_test_v2 AS
SELECT
    l.*,

    CASE
        WHEN TRY_CAST(NULLIF(TRIM(classic_fico), '') AS DOUBLE) = 9999
        THEN NULL
        ELSE TRY_CAST(NULLIF(TRIM(classic_fico), '') AS DOUBLE)
    END AS classic_fico_num,

    CASE
        WHEN TRY_CAST(NULLIF(TRIM(mortgage_insurance_percentage_mi), '') AS DOUBLE) = 999
        THEN NULL
        ELSE TRY_CAST(NULLIF(TRIM(mortgage_insurance_percentage_mi), '') AS DOUBLE)
    END AS mi_percentage_num,

    CASE
        WHEN TRY_CAST(NULLIF(TRIM(number_of_units), '') AS DOUBLE) = 99
        THEN NULL
        ELSE TRY_CAST(NULLIF(TRIM(number_of_units), '') AS DOUBLE)
    END AS number_of_units_num,

    CASE
        WHEN TRY_CAST(NULLIF(TRIM(original_loan_to_value_ltv), '') AS DOUBLE) = 999
        THEN NULL
        ELSE TRY_CAST(NULLIF(TRIM(original_loan_to_value_ltv), '') AS DOUBLE)
    END AS original_ltv_num,

    CASE
        WHEN TRY_CAST(NULLIF(TRIM(original_combined_loan_to_value_cltv), '') AS DOUBLE) = 999
        THEN NULL
        ELSE TRY_CAST(NULLIF(TRIM(original_combined_loan_to_value_cltv), '') AS DOUBLE)
    END AS original_cltv_num,

    CASE
        WHEN TRY_CAST(NULLIF(TRIM(original_debt_to_income_dti_ratio), '') AS DOUBLE) = 999
        THEN NULL
        ELSE TRY_CAST(NULLIF(TRIM(original_debt_to_income_dti_ratio), '') AS DOUBLE)
    END AS original_dti_num,

    TRY_CAST(NULLIF(TRIM(original_upb), '') AS DOUBLE)
        AS original_upb_num,

    TRY_CAST(NULLIF(TRIM(original_interest_rate), '') AS DOUBLE)
        AS original_interest_rate_num,

    TRY_CAST(NULLIF(TRIM(original_loan_term), '') AS DOUBLE)
        AS original_loan_term_num,

    TRY_CAST(NULLIF(TRIM(number_of_borrowers), '') AS DOUBLE)
        AS number_of_borrowers_num

FROM read_parquet('{source_path_sql}') l
""")

# 2. Create data-quality flags
con.execute("""
CREATE OR REPLACE TEMP VIEW cleaned_test_v2_flags AS
SELECT
    *,

    CASE
        WHEN original_ltv_num IS NOT NULL
         AND (original_ltv_num < 6 OR original_ltv_num > 105)
        THEN 1 ELSE 0
    END AS ltv_outside_guide_range,

    CASE
        WHEN original_cltv_num IS NOT NULL
         AND (original_cltv_num < 6 OR original_cltv_num > 200)
        THEN 1 ELSE 0
    END AS cltv_outside_guide_range,

    CASE
        WHEN TRIM(harp_indicator) = 'Y'
         AND original_ltv_num BETWEEN 6 AND 80
        THEN 1 ELSE 0
    END AS harp_oltv_inconsistency,

    CASE
        WHEN TRIM(harp_indicator) = 'Y'
         AND original_ltv_num > 105
        THEN 1 ELSE 0
    END AS harp_ltv_extreme,

    CASE
        WHEN TRIM(harp_indicator) = 'Y'
         AND original_cltv_num > 200
        THEN 1 ELSE 0
    END AS harp_cltv_extreme

FROM cleaned_test_v2
""")

# 3. Write test Parquet
con.execute(f"""
COPY cleaned_test_v2_flags
TO '{output_path_sql}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Created:", clean_test_v2)
print("File exists:", clean_test_v2.exists())

if clean_test_v2.exists():
    print("File size (MB):", round(clean_test_v2.stat().st_size / 1024**2, 2))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Created: E:\DS_project_backup\credit-risk-portfolio\processed\eda\loan_level\loan_level_clean_2012Q4_test_v2.parquet
File exists: True
File size (MB): 14.16


In [39]:
# CELL 10Q — Validate cleaned file and data-quality flags

cleaned_v2_stats = con.execute("""
SELECT
    COUNT(*) AS total_loans,
    COUNT(DISTINCT loan_identifier) AS unique_loans,

    COUNT(*) FILTER (
        WHERE ltv_outside_guide_range = 1
    ) AS ltv_outside_guide_range,

    COUNT(*) FILTER (
        WHERE cltv_outside_guide_range = 1
    ) AS cltv_outside_guide_range,

    COUNT(*) FILTER (
        WHERE harp_oltv_inconsistency = 1
    ) AS harp_oltv_inconsistency,

    COUNT(*) FILTER (
        WHERE harp_ltv_extreme = 1
    ) AS harp_ltv_extreme,

    COUNT(*) FILTER (
        WHERE harp_cltv_extreme = 1
    ) AS harp_cltv_extreme,

    COUNT(*) FILTER (
        WHERE original_ltv_num IS NULL
    ) AS ltv_missing,

    COUNT(*) FILTER (
        WHERE original_cltv_num IS NULL
    ) AS cltv_missing

FROM read_parquet(?)
""", [str(clean_test_v2)]).df()

print("CLEANED DATA VALIDATION")
print(cleaned_v2_stats.to_string(index=False))


# Check that original raw columns are preserved
raw_preservation = con.execute("""
SELECT
    COUNT(*) FILTER (
        WHERE original_loan_to_value_ltv IS NULL
    ) AS raw_ltv_null,

    COUNT(*) FILTER (
        WHERE original_combined_loan_to_value_cltv IS NULL
    ) AS raw_cltv_null,

    COUNT(*) FILTER (
        WHERE original_ltv_num IS NULL
          AND original_loan_to_value_ltv IS NOT NULL
          AND TRIM(original_loan_to_value_ltv) NOT IN ('', '999')
    ) AS unexpected_ltv_conversion_missing,

    COUNT(*) FILTER (
        WHERE original_cltv_num IS NULL
          AND original_combined_loan_to_value_cltv IS NOT NULL
          AND TRIM(original_combined_loan_to_value_cltv) NOT IN ('', '999')
    ) AS unexpected_cltv_conversion_missing

FROM read_parquet(?)
""", [str(clean_test_v2)]).df()

print("\nRAW VALUE PRESERVATION CHECK")
print(raw_preservation.to_string(index=False))

CLEANED DATA VALIDATION
 total_loans  unique_loans  ltv_outside_guide_range  cltv_outside_guide_range  harp_oltv_inconsistency  harp_ltv_extreme  harp_cltv_extreme  ltv_missing  cltv_missing
      592713        592713                    48806                      2523                    77971             48776               2504            8             9

RAW VALUE PRESERVATION CHECK
 raw_ltv_null  raw_cltv_null  unexpected_ltv_conversion_missing  unexpected_cltv_conversion_missing
            0              0                                  0                                   0


In [40]:
# CELL 10R — Batch cleaning for all 28 vintages

import gc
import time

clean_manifest = []
failed_vintages = []

for vintage in VINTAGES:

    start_time = time.time()

    source_file = EDA_DATA_DIR / f"loan_level_{vintage}.parquet"
    output_file = EDA_DATA_DIR / f"loan_level_clean_v2_{vintage}.parquet"

    if not source_file.exists():
        print(f"SKIPPED — Source missing: {vintage}")
        failed_vintages.append({
            "vintage": vintage,
            "error": "Source file missing"
        })
        continue

    source_sql = str(source_file).replace("\\", "/").replace("'", "''")
    output_sql = str(output_file).replace("\\", "/").replace("'", "''")

    try:
        # Build cleaned analytical dataset
        con.execute(f"""
        COPY (
            WITH cleaned AS (
                SELECT
                    l.*,

                    CASE
                        WHEN TRY_CAST(NULLIF(TRIM(classic_fico), '') AS DOUBLE) = 9999
                        THEN NULL
                        ELSE TRY_CAST(NULLIF(TRIM(classic_fico), '') AS DOUBLE)
                    END AS classic_fico_num,

                    CASE
                        WHEN TRY_CAST(NULLIF(TRIM(mortgage_insurance_percentage_mi), '') AS DOUBLE) = 999
                        THEN NULL
                        ELSE TRY_CAST(NULLIF(TRIM(mortgage_insurance_percentage_mi), '') AS DOUBLE)
                    END AS mi_percentage_num,

                    CASE
                        WHEN TRY_CAST(NULLIF(TRIM(number_of_units), '') AS DOUBLE) = 99
                        THEN NULL
                        ELSE TRY_CAST(NULLIF(TRIM(number_of_units), '') AS DOUBLE)
                    END AS number_of_units_num,

                    CASE
                        WHEN TRY_CAST(NULLIF(TRIM(original_loan_to_value_ltv), '') AS DOUBLE) = 999
                        THEN NULL
                        ELSE TRY_CAST(NULLIF(TRIM(original_loan_to_value_ltv), '') AS DOUBLE)
                    END AS original_ltv_num,

                    CASE
                        WHEN TRY_CAST(NULLIF(TRIM(original_combined_loan_to_value_cltv), '') AS DOUBLE) = 999
                        THEN NULL
                        ELSE TRY_CAST(NULLIF(TRIM(original_combined_loan_to_value_cltv), '') AS DOUBLE)
                    END AS original_cltv_num,

                    CASE
                        WHEN TRY_CAST(NULLIF(TRIM(original_debt_to_income_dti_ratio), '') AS DOUBLE) = 999
                        THEN NULL
                        ELSE TRY_CAST(NULLIF(TRIM(original_debt_to_income_dti_ratio), '') AS DOUBLE)
                    END AS original_dti_num,

                    TRY_CAST(NULLIF(TRIM(original_upb), '') AS DOUBLE)
                        AS original_upb_num,

                    TRY_CAST(NULLIF(TRIM(original_interest_rate), '') AS DOUBLE)
                        AS original_interest_rate_num,

                    TRY_CAST(NULLIF(TRIM(original_loan_term), '') AS DOUBLE)
                        AS original_loan_term_num,

                    TRY_CAST(NULLIF(TRIM(number_of_borrowers), '') AS DOUBLE)
                        AS number_of_borrowers_num

                FROM read_parquet('{source_sql}') l
            )

            SELECT
                *,

                CASE
                    WHEN original_ltv_num IS NOT NULL
                     AND (original_ltv_num < 6 OR original_ltv_num > 105)
                    THEN 1 ELSE 0
                END AS ltv_outside_guide_range,

                CASE
                    WHEN original_cltv_num IS NOT NULL
                     AND (original_cltv_num < 6 OR original_cltv_num > 200)
                    THEN 1 ELSE 0
                END AS cltv_outside_guide_range,

                CASE
                    WHEN TRIM(harp_indicator) = 'Y'
                     AND original_ltv_num BETWEEN 6 AND 80
                    THEN 1 ELSE 0
                END AS harp_oltv_inconsistency,

                CASE
                    WHEN TRIM(harp_indicator) = 'Y'
                     AND original_ltv_num > 105
                    THEN 1 ELSE 0
                END AS harp_ltv_extreme,

                CASE
                    WHEN TRIM(harp_indicator) = 'Y'
                     AND original_cltv_num > 200
                    THEN 1 ELSE 0
                END AS harp_cltv_extreme

            FROM cleaned

        ) TO '{output_sql}'
        (FORMAT PARQUET, COMPRESSION ZSTD)
        """)

        # Validate output
        stats = con.execute("""
            SELECT
                COUNT(*) AS total_loans,
                COUNT(DISTINCT loan_identifier) AS unique_loans,
                COUNT(*) FILTER (
                    WHERE ltv_outside_guide_range = 1
                ) AS ltv_outside_guide_range,
                COUNT(*) FILTER (
                    WHERE cltv_outside_guide_range = 1
                ) AS cltv_outside_guide_range,
                COUNT(*) FILTER (
                    WHERE harp_oltv_inconsistency = 1
                ) AS harp_oltv_inconsistency,
                COUNT(*) FILTER (
                    WHERE harp_ltv_extreme = 1
                ) AS harp_ltv_extreme,
                COUNT(*) FILTER (
                    WHERE harp_cltv_extreme = 1
                ) AS harp_cltv_extreme
            FROM read_parquet(?)
        """, [str(output_file)]).fetchone()

        elapsed = round(time.time() - start_time, 2)

        clean_manifest.append({
            "vintage": vintage,
            "total_loans": stats[0],
            "unique_loans": stats[1],
            "ltv_outside_guide_range": stats[2],
            "cltv_outside_guide_range": stats[3],
            "harp_oltv_inconsistency": stats[4],
            "harp_ltv_extreme": stats[5],
            "harp_cltv_extreme": stats[6],
            "file_size_mb": round(output_file.stat().st_size / 1024**2, 2),
            "processing_seconds": elapsed,
            "status": "SUCCESS"
        })

        print(
            f"SUCCESS {vintage}: "
            f"{stats[0]:,} loans | "
            f"{elapsed:.1f}s"
        )

    except Exception as e:
        failed_vintages.append({
            "vintage": vintage,
            "error": str(e)
        })
        print(f"FAILED {vintage}: {e}")

    gc.collect()


# Save manifests
clean_manifest_df = pd.DataFrame(clean_manifest)

clean_manifest_file = EDA_DIR / "loan_level_cleaning_manifest_v2.csv"
clean_manifest_df.to_csv(clean_manifest_file, index=False)

failed_manifest_file = EDA_DIR / "loan_level_cleaning_failures_v2.csv"
pd.DataFrame(failed_vintages).to_csv(failed_manifest_file, index=False)

print("\n" + "=" * 60)
print("BATCH CLEANING SUMMARY")
print("=" * 60)

print("Successful vintages:", len(clean_manifest))
print("Failed vintages:", len(failed_vintages))
print(
    "Total loans:",
    f"{clean_manifest_df['total_loans'].sum():,}"
    if not clean_manifest_df.empty else 0
)
print("Manifest:", clean_manifest_file)

SUCCESS 2006Q1: 298,574 loans | 1.6s
SUCCESS 2006Q2: 308,208 loans | 1.6s
SUCCESS 2006Q3: 283,975 loans | 1.4s
SUCCESS 2006Q4: 302,787 loans | 1.4s
SUCCESS 2007Q1: 302,331 loans | 1.6s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SUCCESS 2007Q2: 354,055 loans | 2.3s
SUCCESS 2007Q3: 284,980 loans | 1.7s
SUCCESS 2007Q4: 296,826 loans | 1.8s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SUCCESS 2008Q1: 421,635 loans | 2.3s
SUCCESS 2008Q2: 370,982 loans | 1.9s
SUCCESS 2008Q3: 221,752 loans | 1.8s
SUCCESS 2008Q4: 218,777 loans | 1.7s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (21 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SUCCESS 2012Q3: 563,453 loans | 3.0s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SUCCESS 2012Q4: 592,713 loans | 3.2s

BATCH CLEANING SUMMARY
Successful vintages: 28
Failed vintages: 0
Total loans: 11,128,442
Manifest: E:\DS_project_backup\credit-risk-portfolio\processed\eda\loan_level_cleaning_manifest_v2.csv


In [41]:
# CELL 10S — Batch-level integrity validation

clean_files = sorted(
    EDA_DATA_DIR.glob("loan_level_clean_v2_*.parquet")
)

clean_files = [
    f for f in clean_files
    if "_test_" not in f.stem
]

print("Cleaned files:", len(clean_files))

assert len(clean_files) == 28, (
    f"Expected 28 files, found {len(clean_files)}"
)

batch_validation = con.execute("""
SELECT
    COUNT(*) AS total_loans,
    COUNT(DISTINCT loan_identifier) AS unique_loans,

    SUM(ltv_outside_guide_range) AS ltv_outside_guide_range,
    SUM(cltv_outside_guide_range) AS cltv_outside_guide_range,
    SUM(harp_oltv_inconsistency) AS harp_oltv_inconsistency,
    SUM(harp_ltv_extreme) AS harp_ltv_extreme,
    SUM(harp_cltv_extreme) AS harp_cltv_extreme

FROM read_parquet(?)
""", [ [str(f) for f in clean_files] ]).df()

print("\nBATCH VALIDATION")
print(batch_validation.to_string(index=False))

print("\nMANIFEST VALIDATION")
print(clean_manifest_df[
    ["vintage", "total_loans", "unique_loans", "status"]
].to_string(index=False))

# Integrity assertions
assert clean_manifest_df["status"].eq("SUCCESS").all()
assert clean_manifest_df["total_loans"].sum() == EXPECTED_LOANS
assert clean_manifest_df["unique_loans"].sum() == EXPECTED_LOANS

assert (
    batch_validation.iloc[0]["total_loans"]
    == EXPECTED_LOANS
)

assert (
    batch_validation.iloc[0]["unique_loans"]
    == EXPECTED_LOANS
)

print("\nALL BATCH INTEGRITY CHECKS PASSED.")

Cleaned files: 28


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


BATCH VALIDATION
 total_loans  unique_loans  ltv_outside_guide_range  cltv_outside_guide_range  harp_oltv_inconsistency  harp_ltv_extreme  harp_cltv_extreme
    11128442      11128442             274,067.0000               12,912.0000             940,592.0000      273,926.0000        12,813.0000

MANIFEST VALIDATION
vintage  total_loans  unique_loans  status
 2006Q1       298574        298574 SUCCESS
 2006Q2       308208        308208 SUCCESS
 2006Q3       283975        283975 SUCCESS
 2006Q4       302787        302787 SUCCESS
 2007Q1       302331        302331 SUCCESS
 2007Q2       354055        354055 SUCCESS
 2007Q3       284980        284980 SUCCESS
 2007Q4       296826        296826 SUCCESS
 2008Q1       421635        421635 SUCCESS
 2008Q2       370982        370982 SUCCESS
 2008Q3       221752        221752 SUCCESS
 2008Q4       218777        218777 SUCCESS
 2009Q1       621542        621542 SUCCESS
 2009Q2       716665        716665 SUCCESS
 2009Q3       443064        443064 S